In [15]:
import os

os.environ["KERAS_BACKEND"] = "jax"

import random
import io

import kagglehub
from pathlib import Path
import grain.python as grain
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
import keras

In [16]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [17]:
KH_DATA_PATH = Path(
    kagglehub.dataset_download("zulkarnainsaurav/four-genre-movie-poster-images")
)
DATA_DIR = Path(KH_DATA_PATH) / "four_genre_posters" / "four_genre_posters"

Using Colab cache for faster access to the 'four-genre-movie-poster-images' dataset.


In [18]:
CLASSES = ["Action", "Comedy", "Horror", "Romance"]
N_CLASSES = len(CLASSES)

IMG_SIZE = (380, 562)  # We are going to use the original sz
BATCH_SIZE = 64

In [19]:
def gen(loader):
    for batch in loader:
        yield batch["image"], batch["label"]

In [20]:
class ImageProcessor(grain.MapTransform):
    def map(self, data):
        with open(data["path"], "rb") as f:
            img = Image.open(io.BytesIO(f.read())).convert("RGB")

        img_array = np.array(img).astype(np.float32) / 255.0

        label = data["label"]
        one_hot = np.zeros(N_CLASSES, dtype=np.float32)
        one_hot[label] = 1.0

        return {"image": img_array, "label": one_hot}


transformations = [ImageProcessor(), grain.Batch(BATCH_SIZE)]

In [21]:
def create_loader(files, shuffle=False):
    sampler = grain.IndexSampler(
        len(files), shard_options=grain.NoSharding(), shuffle=shuffle, seed=SEED
    )
    source = grain.MapDataset.source(files)
    return grain.DataLoader(
        data_source=source, operations=transformations, sampler=sampler
    )

In [22]:
ds = []
for i, class_name in enumerate(CLASSES):
    class_dir = DATA_DIR / class_name
    for img_path in class_dir.glob("*.jpg"):
        ds.append({"path": str(img_path), "label": i})

In [23]:
train_f, temp_f = train_test_split(ds, test_size=0.3, stratify=[d["label"] for d in ds])
val_f, test_f = train_test_split(
    temp_f, test_size=0.5, stratify=[d["label"] for d in temp_f]
)

In [24]:
train_loader = create_loader(train_f, shuffle=True)
val_loader = create_loader(val_f)
test_loader = create_loader(test_f)

In [25]:
train_steps = len(train_f) // BATCH_SIZE
val_steps = len(val_f) // BATCH_SIZE

In [26]:
model = keras.Sequential(
    [
        keras.layers.Input(shape=(*IMG_SIZE, 3)),
        keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
        keras.layers.MaxPooling2D(),
        keras.layers.Conv2D(64, 3, activation="relu"),
        keras.layers.GlobalAveragePooling2D(),
        keras.layers.Dense(N_CLASSES, activation="softmax"),
    ]
)

In [27]:
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

In [28]:
m_callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=10, restore_best_weights=True
    ),
    keras.callbacks.ModelCheckpoint(
        filepath="best_model.keras",
        monitor="val_loss",
        save_best_only=True,
    ),
    keras.callbacks.CSVLogger(filename="train_log.csv", append=False),
]

In [29]:
train_hist = model.fit(
    gen(train_loader),
    steps_per_epoch=train_steps,
    validation_data=gen(val_loader),
    validation_steps=val_steps,
    epochs=10,
    callbacks=m_callbacks,
)

Epoch 1/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 20s 1s/step - accuracy: 0.2545 - loss: 1.3750 - val_accuracy: 0.3021 - val_loss: 1.3547
Epoch 2/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 9s 649ms/step - accuracy: 0.3850 - loss: 1.3475 - val_accuracy: 0.4167 - val_loss: 1.3204
Epoch 3/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 9s 638ms/step - accuracy: 0.4286 - loss: 1.2937 - val_accuracy: 0.4010 - val_loss: 1.2569
Epoch 4/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 9s 640ms/step - accuracy: 0.4330 - loss: 1.2473 - val_accuracy: 0.4479 - val_loss: 1.2142
Epoch 5/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 10s 696ms/step - accuracy: 0.4431 - loss: 1.1969 - val_accuracy: 0.4948 - val_loss: 1.1549
Epoch 6/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 9s 650ms/step - accuracy: 0.4944 - loss: 1.1695 - val_accuracy: 0.4740 - val_loss: 1.1405
Epoch 7/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 9s 647ms/step - accuracy: 0.4397 - loss: 1.1740 - val_accuracy: 0.4531 - val_loss: 1.1578
Epoch 8/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 9s 633ms/step - accuracy: 0.4688 - loss: 1.1583 - val_accuracy: 0.5